In [1]:
print("hello COT!")

hello COT!


In [31]:
from dataclasses import dataclass
from openai import OpenAI
import os

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests accross all inference providers """

    name: str
    env_var: str
    is_true: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("Groq","GROQ_API_KEY",True,"https://api.groq.com/openai/v1","openai/gpt-oss-120b"),
    Provider("OpenRouter","OPENROUTER_API_KEY",True,"https://openrouter.ai/api/v1","stealth/space-bunny-alpha")
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")


def build_client(provider: Provider) -> OpenAI:

    api_key = os.getenv(provider.env_var)


    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.


In [32]:
def model_reply(prompt: str) -> str :

    provider = select_provider()
    print(f"Using {provider.name} provider and {provider.model} model")
    client = build_client(provider)

    reply = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return reply.choices[0].message.content

## Zero Shot COT

In [33]:
ZERO_SHOT_Suffix = "\n\nLet's think step by step"

In [34]:
def zero_shot_prompt(question: str) -> str:
    f"""This is function takes question and attach zero shot suffix!"""
    return f"Question: {question}{ZERO_SHOT_Suffix}"

In [35]:
Question = (
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)

In [36]:
print(zero_shot_prompt(Question))

Question: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?

Let's think step by step


In [37]:
llm_reply = model_reply(zero_shot_prompt(Question))

print(llm_reply)

Using Groq provider and openai/gpt-oss-120b model
**Step‑by‑step solution**

1. **Start with what Roger already has**  
   He has **5 tennis balls**.

2. **Find out how many new balls he buys**  
   - He buys **2 cans**.  
   - Each can contains **3 tennis balls**.  
   - Total new balls = \(2 \text{ cans} \times 3 \text{ balls per can} = 6\) balls.

3. **Add the new balls to the original amount**  
   \[
   5 \text{ (original)} + 6 \text{ (new)} = 11 \text{ tennis balls}
   \]

---




## Few Shot COT

In [39]:
Few_shot_cot_examples = """
You are an expert logic and math reasoning assistant. Solve the final question by showing a clear, step-by-step reasoning process before giving the final answer, matching the exact format of the examples provided below.

### Example 1
Question: I went to the market and bought 10 apples. I gave 2 apples to my neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples do I have left?
Reasoning:
1. I started with 10 apples.
2. I gave away 2 to the neighbor and 2 to the repairman: 10 - 2 - 2 = 6 apples left.
3. I bought 5 more apples: 6 + 5 = 11 apples.
4. I ate 1 apple: 11 - 1 = 10 apples.
Answer: 10

### Example 2
Question: A clothing store is having a sale where all shirts are 20% off. If a shirt originally costs $40, and there is a flat $5 shipping fee per order, how much will a customer pay in total for one shirt?
Reasoning:
1. Find the discount amount on the shirt: 20% of $40 = 0.20 * 40 = $8.
2. Calculate the sale price of the shirt: $40 - $8 = $32.
3. Add the flat shipping fee to the sale price: $32 + $5 = $37.
Answer: $37

### Example 3
Question: John runs faster than Mark. Mark runs faster than Sam. If Sam can run a mile in 8 minutes, is John's mile time likely more or less than 8 minutes?
Reasoning:
1. The question establishes a speed hierarchy: John > Mark > Sam.
2. Since Sam takes 8 minutes to run a mile, anyone faster than Sam will take less time to complete the same distance.
3. Mark is faster than Sam, so Mark takes less than 8 minutes.
4. John is faster than Mark, meaning John is also faster than Sam.
5. Therefore, John will take the least amount of time.
Answer: Less than 8 minutes

"""

In [41]:
def Few_Shot_Prompt(question: str) -> str:
    """This function is going to take few shot example and create final prompt"""
    return f"{Few_shot_cot_examples}\n\n #Final Question\n{question}"

In [43]:
Few_Shot_Question = (
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)

In [47]:
few_shot_cot_llmResponse = model_reply(Few_Shot_Prompt(Few_Shot_Question))

print(few_shot_cot_llmResponse)

Using Groq provider and openai/gpt-oss-120b model
**Reasoning:**
1. Roger starts with 5 tennis balls.  
2. He buys 2 cans, and each can contains 3 tennis balls.  
   - Total balls from the cans = 2 cans × 3 balls per can = 6 balls.  
3. Add the balls from the cans to the balls he already has:  
   - 5 balls + 6 balls = 11 balls.

**Answer:** 11
